# Angelina Benchmark Review

## tl;dr

Baseline v1 hanya mencapai top-1 22,26% pada benchmark eksternal Angelina. Kandidat eksperimen mencapai 93,11% pada 9.625 crop dari `uploaded/test.txt`, melewati target karakter internal 75%, dan sekarang dipasang sebagai v2 hanya untuk development lokal. Status production tetap `NO-GO` karena lisensi dataset tidak eksplisit, cakupan V/Y lemah, dan evaluasi memakai bounding box ground-truth alih-alih ROI kamera native.

## Context & Methods

Notebook review ini membaca evidence yang dihasilkan oleh `training/scripts/benchmark_angelina.py` dan `training/scripts/train_angelina_candidate.py`. Training tidak dijalankan ulang di sini agar review tetap cepat; kedua script menyediakan resep yang dapat dijalankan ulang, tetapi hash hasil retraining tidak dijamin identik lintas environment.

### Key Assumptions

- Revision Angelina dikunci ke `e13038d483bb58e17046d640e078dc4768276e9b`.
- Target 75% berlaku untuk top-1 semua crop karakter, bukan hanya prediksi yang lolos threshold.
- `uploaded/test.txt` tidak digunakan untuk training atau pemilihan epoch kandidat ini, tetapi kini sudah memengaruhi promosi sehingga bukan lagi pristine holdout untuk kandidat berikutnya.
- Hasil offline tidak digunakan sebagai klaim akurasi kamera native atau status production-ready.

## Data

### 1. Load saved evidence

In [1]:
import json
from pathlib import Path

repo_root = Path.cwd().resolve()
if not (repo_root / 'training').is_dir():
    raise RuntimeError(f'Jalankan notebook dari root repository, bukan {repo_root}')

baseline_path = repo_root / 'training/evidence/braille-cell-classifier-v1/angelina_external_benchmark.json'
experiment_path = repo_root / 'training/experiments/angelina-2026-09-04/RESULTS.json'
baseline = json.loads(baseline_path.read_text(encoding='utf-8'))
experiment = json.loads(experiment_path.read_text(encoding='utf-8'))
print(f"Baseline evidence: {baseline_path.relative_to(repo_root)}")
print(f"Experiment evidence: {experiment_path.relative_to(repo_root)}")

Baseline evidence: training\evidence\braille-cell-classifier-v1\angelina_external_benchmark.json
Experiment evidence: training\experiments\angelina-2026-09-04\RESULTS.json


### 2. Validate provenance and population

In [2]:
expected_revision = 'e13038d483bb58e17046d640e078dc4768276e9b'
assert baseline['dataset']['revision'] == expected_revision
assert experiment['dataset']['revision'] == expected_revision
assert baseline['dataset']['includedAzCellCount'] == 69056
assert experiment['dataset']['externalTestPageCount'] == 50
assert experiment['dataset']['externalTestAzCellCount'] == 9625
assert baseline['dataset']['licenseStatus'].startswith('no_explicit_license')

population = {
    'baseline_pages': baseline['dataset']['sourcePageCount'],
    'baseline_a_z_cells': baseline['dataset']['includedAzCellCount'],
    'external_test_pages': experiment['dataset']['externalTestPageCount'],
    'external_test_a_z_cells': experiment['dataset']['externalTestAzCellCount'],
    'external_test_v_support': experiment['candidate']['externalTestVSupport'],
    'external_test_y_support': experiment['candidate']['externalTestYSupport'],
}
population

{'baseline_pages': 290,
 'baseline_a_z_cells': 69056,
 'external_test_pages': 50,
 'external_test_a_z_cells': 9625,
 'external_test_v_support': 9,
 'external_test_y_support': 0}

## Results

### 3. Reconcile target and promotion decision

In [3]:
target = 0.75
baseline_accuracy = baseline['metrics']['top1Accuracy']
candidate_accuracy = experiment['candidate']['externalTestTop1Accuracy']
assert baseline_accuracy < target
assert candidate_accuracy >= target
assert experiment['candidate']['internal75PercentTargetPassed'] is True
assert experiment['decision']['promotedToApplication'] is True

summary = {
    'target': target,
    'baseline_top1_accuracy': baseline_accuracy,
    'candidate_external_test_top1_accuracy': candidate_accuracy,
    'candidate_coverage_at_0_90': experiment['candidate']['externalTestCoverageAt090'],
    'candidate_accepted_accuracy': experiment['candidate']['externalTestAcceptedAccuracy'],
    'candidate_v_recall': experiment['candidate']['externalTestVRecall'],
    'promoted_to_application': experiment['decision']['promotedToApplication'],
}
summary

{'target': 0.75,
 'baseline_top1_accuracy': 0.2226453892493049,
 'candidate_external_test_top1_accuracy': 0.9311168831168831,
 'candidate_coverage_at_0_90': 0.7274805194805195,
 'candidate_accepted_accuracy': 0.9930019994287347,
 'candidate_v_recall': 0.1111111111111111,
 'promoted_to_application': True}

## Takeaways

Kandidat eksperimen memenuhi target agregat 75% pada external test dan telah dipasang sebagai model aktif hanya untuk development lokal. Recall V masih 11,11% dari 9 sampel dan Y tidak tersedia pada external test. Hak penggunaan Angelina untuk redistribusi model juga belum eksplisit; self-test post-crop sudah lulus di emulator, tetapi camera golden parity fisik belum diuji. Keputusan yang didukung evidence adalah mempertahankan v2 sebagai `unverified_candidate` dan menjaga status production `NO-GO`.